# How the forecaster was built

The path from *"does consumption depend on the weather"* to a per-household daily
forecast, in the order the work actually went.

**What is here:** the two naive references that set the acceptance bar, the daily
temperature response, the fitted form, forward chaining, and one household end to end.

**What is not here:** the hourly rungs, the rejected specifications, and the window
variants that were not selected. This is the critical path only.

**Sources.** Two files, both already on disk. Nothing is fitted in this notebook —
the coefficients are read and applied.

| file | what |
|---|---|
| `f3daily/panels/f3daily_panel.parquet` | one row per household per local day: `kwh_day`, `temp_mean`, fold keys |
| `f3d/results/f3d_window_coefficients.parquet` | the model: `a`, `b1`, `b2` per household per fold |

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# walk up until the project root is found, so the notebook runs whether the
# kernel starts in outputs/r7/ or in the project root
ROOT = Path.cwd()
while not (ROOT / "heapo_data").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

PANEL = ROOT / "Forecast_consumption_cycle/outputs/f3daily/panels/f3daily_panel.parquet"
COEF  = ROOT / "Forecast_consumption_cycle/outputs/f3d/results/f3d_window_coefficients.parquet"

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

In [ ]:
d = pd.read_parquet(PANEL, columns=["Household_ID", "local_date", "kwh_day", "temp_mean",
                                    "moy", "ym", "month_idx", "complete_day"])
d["local_date"] = pd.to_datetime(d["local_date"])
d = d[d["complete_day"]]

print(len(d), "complete household-days")
print(d["Household_ID"].nunique(), "households")
print(d["local_date"].min().date(), "to", d["local_date"].max().date())

---
## 1. Every day, every household

One hexagon = many household-days. The colour is how many.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
hb = ax.hexbin(d["temp_mean"], d["kwh_day"], gridsize=60, bins="log",
               cmap="Blues", mincnt=1)
ax.set_ylim(0, d["kwh_day"].quantile(0.999))
ax.set_xlabel("daily mean temperature (C)")
ax.set_ylabel("daily consumption (kWh)")
ax.set_title("%s household-days" % format(len(d), ","))
fig.colorbar(hb, ax=ax, label="days (log scale)")
plt.show()

print("correlation, all days pooled: %.3f" % d["kwh_day"].corr(d["temp_mean"]))

Consumption falls as it gets warmer, and it flattens out somewhere above 15 C —
once the heating stops there is a floor left over that the weather does not move.

That bend is the whole model. But look how wide the cloud is: at 5 C a household
might use 10 kWh or 60. Pooled like this the correlation is only about -0.55.

---
## 2. The same thing by month

Average each household within each calendar month, then plot the monthly means.

In [ ]:
m = (d.groupby(["Household_ID", "ym"])
       .agg(kwh=("kwh_day", "mean"), temp=("temp_mean", "mean"), n=("kwh_day", "size"))
       .reset_index())
m = m[m["n"] >= 25]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(m["temp"], m["kwh"], s=5, alpha=0.12, color="#2E86C1", linewidths=0)
ax.set_ylim(0, m["kwh"].quantile(0.999))
ax.set_xlabel("monthly mean temperature (C)")
ax.set_ylabel("monthly mean daily consumption (kWh)")
ax.set_title("%s household-months (>= 25 days each)" % format(len(m), ","))
plt.show()

print("correlation, monthly means pooled: %.3f" % m["kwh"].corr(m["temp"]))

Barely tighter than the daily plot. That is not what averaging is supposed to do,
and the reason matters.

---
## 3. Why the pooled picture is misleading

The cloud is wide because households sit at different levels. A big house at 15 C
uses more than a small house at 0 C, so pooling them smears the relationship.

Measure it inside each household instead.

In [ ]:
daily_r = {}
for hid, g in d.groupby("Household_ID"):
    daily_r[hid] = g["kwh_day"].corr(g["temp_mean"])
daily_r = pd.Series(daily_r)

monthly_r = {}
for hid, g in m.groupby("Household_ID"):
    if len(g) >= 6:
        monthly_r[hid] = g["kwh"].corr(g["temp"])
monthly_r = pd.Series(monthly_r)

print("correlation of consumption with temperature")
print()
print("  pooled over all households     daily %6.3f    monthly %6.3f"
      % (d["kwh_day"].corr(d["temp_mean"]), m["kwh"].corr(m["temp"])))
print("  median WITHIN one household    daily %6.3f    monthly %6.3f"
      % (daily_r.median(), monthly_r.median()))

Inside a single household the relationship is strong, and monthly averaging does
tighten it. Pooling was hiding it.

**This is why the model is fitted per household** — one set of coefficients each,
never one fit for the fleet.

---
## 4. The bar: what you get with no weather at all

Before fitting anything to temperature, you need to know what a model that ignores
temperature already achieves. Two references, neither of which sees the weather:

- **R1, seasonal mean** — predict this day with the average of the same calendar
  month in previous years. Only months already seen count.
- **R2, persistence** — predict this day with the same day one week earlier.

From here on: household **756811**. A control (never inspected), HIGH fit-quality
band, 847 usable days.

In [ ]:
H = 756811

h = d[d["Household_ID"] == H].sort_values("local_date").copy()

g = (h.groupby(["moy", "month_idx"])["kwh_day"].agg(["sum", "size"])
       .reset_index().sort_values(["moy", "month_idx"]))
by_moy = g.groupby("moy")
g["prev_sum"] = by_moy["sum"].cumsum() - g["sum"]
g["prev_n"] = by_moy["size"].cumsum() - g["size"]
g["r1"] = np.where(g["prev_n"] > 0, g["prev_sum"] / g["prev_n"], np.nan)
h = h.merge(g[["moy", "month_idx", "r1"]], on=["moy", "month_idx"], how="left")

week_ago = h[["local_date", "kwh_day"]].copy()
week_ago["local_date"] = week_ago["local_date"] + pd.Timedelta(days=7)
week_ago.columns = ["local_date", "r2"]
h = h.merge(week_ago, on="local_date", how="left")

print(len(h), "days for household", H)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(h["local_date"], h["kwh_day"], color="#3F4C55", lw=0.9, label="actual")
ax.plot(h["local_date"], h["r1"], color="#B33025", lw=1.2, label="R1 seasonal mean")
ax.set_xlabel("date")
ax.set_ylabel("daily consumption (kWh)")
ax.set_title("Household %d - actual against the seasonal mean" % H)
ax.legend()
plt.show()

R1 gets the shape of the year right and misses every individual cold snap, because
it has no idea what the weather did on any particular day.

---
## 5. The fitted form

Two straight lines meeting at 15 C:

    predicted = a + b1 * max(0, 15 - T) + b2 * max(0, T - 15)

`b1` is the heating slope, how much consumption rises per degree below the knot.
`b2` is the slope above it. `a` is the level.

The knot is fixed at 15 C for every household. **It is not the degree-day base** —
that is 12 C and is a separate parameter used elsewhere.

In [ ]:
c = pd.read_parquet(COEF)
c = c[(c["variant"] == "W2_hl6") & (c["Household_ID"] == H)][["month_idx", "ym", "a", "b1", "b2", "n_prior_days"]]

print(len(c), "folds fitted for household", H)
print()
print(c.head(5).to_string(index=False))

In [ ]:
last = c.iloc[-1]

t = np.linspace(h["temp_mean"].min(), h["temp_mean"].max(), 200)
fit = last["a"] + last["b1"] * np.maximum(0, 15 - t) + last["b2"] * np.maximum(0, t - 15)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(h["temp_mean"], h["kwh_day"], s=9, alpha=0.35, color="#3F4C55", linewidths=0)
ax.plot(t, fit, color="#2E86C1", lw=2.2, label="fitted, fold %d" % last["month_idx"])
ax.axvline(15, color="#767F85", ls="--", lw=1, label="knot at 15 C")
ax.set_xlabel("daily mean temperature (C)")
ax.set_ylabel("daily consumption (kWh)")
ax.set_title("Household %d" % H)
ax.legend()
plt.show()

print("a  = %8.3f  kWh at the knot" % last["a"])
print("b1 = %8.3f  kWh per degree below 15 C" % last["b1"])
print("b2 = %8.3f  kWh per degree above 15 C" % last["b2"])

---
## 6. Forward chaining, and how old days are weighted

The model is refitted every month. Train on everything up to month M, predict month
M+1, step forward. A household needs 12 months of training before the first
prediction, so fold 12 is the first one that produces anything.

The window never discards a day, but it does not treat every day equally either.
The `hl6` in `W2_hl6` is a **half-life of 6 months**. Each training day is weighted

    w = 2 ** (-age_in_months / 6)

and the fit is weighted least squares. A day from 6 months ago counts half as much
as today, 12 months ago a quarter, 24 months ago a sixteenth. That is the whole
difference between this variant and a plain expanding fit: the window grows, the
influence of its oldest days shrinks.

In [ ]:
age = np.arange(0, 37)
for hl, style in [(6, "-"), (12, "--"), (24, ":")]:
    ax = plt.gca()
    ax.plot(age, 2.0 ** (-age / hl), style, label="half-life %d months" % hl)

plt.gcf().set_size_inches(8, 4)
plt.axhline(0.5, color="#767F85", lw=0.8)
plt.axvline(6, color="#767F85", lw=0.8)
plt.xlabel("age of the training day (months)")
plt.ylabel("weight in the fit")
plt.title("W2_hl6 uses the solid line")
plt.legend()
plt.show()

for a in [0, 6, 12, 24]:
    print("a day %2d months old carries weight %.3f" % (a, 2.0 ** (-a / 6)))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for i, r in c.iterrows():
    ax.barh(r["month_idx"], r["month_idx"], left=0, height=0.7, color="#7BAAD0")
    ax.barh(r["month_idx"], 1, left=r["month_idx"], height=0.7, color="#B33025")

ax.set_xlabel("month index")
ax.set_ylabel("fold")
ax.set_title("Household %d - training window (blue) and the month it predicts (red)" % H)
ax.invert_yaxis()
plt.show()

print("first fold trains on", int(c["n_prior_days"].iloc[0]), "days")
print("last  fold trains on", int(c["n_prior_days"].iloc[-1]), "days")

---
## 7. Forecast against actual

Join each day to the coefficients of its own fold and evaluate. Days before fold 12
get nothing, which is correct — no model covered them yet.

In [ ]:
h = h.merge(c[["month_idx", "a", "b1", "b2", "n_prior_days"]], on="month_idx", how="left")
h["pred"] = h["a"] + h["b1"] * np.maximum(0, 15 - h["temp_mean"]) + h["b2"] * np.maximum(0, h["temp_mean"] - 15)

test = h[h["pred"].notna()]

print("training days, never predicted :", int(h["pred"].isna().sum()))
print("predicted days                 :", len(test))
print("folds                          :", test["month_idx"].nunique())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(test["local_date"], test["kwh_day"], color="#3F4C55", lw=0.9, label="actual")
ax.plot(test["local_date"], test["pred"], color="#2E86C1", lw=1.3, label="forecast")
ax.set_xlabel("date")
ax.set_ylabel("daily consumption (kWh)")
ax.set_title("Household %d - forecast against actual, test days only" % H)
ax.legend()
plt.show()

In [ ]:
def score(actual, pred):
    ok = actual.notna() & pred.notna()
    a, p = actual[ok], pred[ok]
    cvrmse = 100 * np.sqrt(((p - a) ** 2).mean()) / a.mean()
    nmbe = 100 * (p - a).sum() / a.sum()
    return cvrmse, nmbe, int(ok.sum())

print("household %d, daily, test days only" % H)
print()
print("%-20s %10s %10s %8s" % ("", "CV(RMSE)", "NMBE", "n"))
for name, col in [("R1 seasonal mean", "r1"), ("R2 persistence", "r2"), ("W2_hl6 weather", "pred")]:
    cv, nb, n = score(test["kwh_day"], test[col])
    print("%-20s %9.2f%% %9.2f%% %8d" % (name, cv, nb, n))

The weather model is more accurate than both references and less biased than R1.

Note the split between the two references: R1 is the more accurate one but runs
**+6% high**, R2 is almost unbiased but far less accurate. That asymmetry is why
the acceptance bar demands accuracy *and* low bias, not one of them.

---
## 8. The training days behind each forecast

Every fold is a different model, fitted on more data than the one before it.
The error does not fall as that training window grows.

In [ ]:
per_fold = []
for mi, grp in test.groupby("month_idx"):
    cv, nb, n = score(grp["kwh_day"], grp["pred"])
    per_fold.append({"fold": mi,
                     "month": int(grp["ym"].iloc[0]),
                     "training_days": int(grp["n_prior_days"].iloc[0]),
                     "test_days": n,
                     "cvrmse": round(cv, 1),
                     "nmbe": round(nb, 1)})

per_fold = pd.DataFrame(per_fold)
print(per_fold.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(per_fold["training_days"], per_fold["cvrmse"], marker="o", color="#2E86C1")
ax.set_xlabel("training days available to that fold")
ax.set_ylabel("CV(RMSE) of that fold (%)")
ax.set_title("Household %d - fold error against training days" % H)
plt.show()

---
## 9. The five households on the dashboard

The demo dashboard serves five households. Same arithmetic as above, applied to
each: join every day to its own fold's coefficients, then score the days that got
a prediction.

In [ ]:
DASH = [512280, 86109, 1037610, 7873107, 6870806]

panel = pd.read_parquet(PANEL, columns=["Household_ID", "local_date", "kwh_day",
                                        "temp_mean", "month_idx", "complete_day"],
                        filters=[("Household_ID", "in", DASH)])
panel["local_date"] = pd.to_datetime(panel["local_date"])
panel = panel[panel["complete_day"]]

coef = pd.read_parquet(COEF)
coef = coef[(coef["variant"] == "W2_hl6") & (coef["Household_ID"].isin(DASH))]

rows = []
fits = {}
for hid in DASH:
    one = panel[panel["Household_ID"] == hid].sort_values("local_date")
    one = one.merge(coef[coef["Household_ID"] == hid][["month_idx", "a", "b1", "b2"]],
                    on="month_idx", how="left")
    one["pred"] = (one["a"] + one["b1"] * np.maximum(0, 15 - one["temp_mean"])
                   + one["b2"] * np.maximum(0, one["temp_mean"] - 15))
    scored = one[one["pred"].notna()]
    fits[hid] = scored
    cv, nb, n = score(scored["kwh_day"], scored["pred"])
    rows.append({"household": hid, "training_days": int(one["pred"].isna().sum()),
                 "predicted_days": n, "folds": scored["month_idx"].nunique(),
                 "cvrmse": round(cv, 1), "nmbe": round(nb, 1)})

print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(len(DASH), 1, figsize=(11, 13))
for ax, hid in zip(axes, DASH):
    s = fits[hid]
    ax.plot(s["local_date"], s["kwh_day"], color="#3F4C55", lw=0.8, label="actual")
    ax.plot(s["local_date"], s["pred"], color="#2E86C1", lw=1.2, label="forecast")
    ax.set_title("household %d" % hid, fontsize=10, loc="left")
    ax.set_ylabel("kWh/day")
axes[0].legend(loc="upper right")
plt.tight_layout()
plt.show()

**Household 6870806 is the one to be careful with.** It has PV, and PV export is not
recorded at 15-minute resolution, so its `kwh_day` is *grid import*, not
consumption. Its self-consumption is invisible to the meter and is itself
weather-driven, so it correlates with the very regressor the model uses. Whatever
score it gets is a score against the wrong number. That is a limit of the data, not
of the fit, which is why the dashboard labels every one of its dates
`pv_contaminated` instead of reporting an error for it.

---
## 10. What the model does and does not contain

Worth being explicit, because the form is smaller than people expect.

There is **no time term anywhere in the equation**. No trend, no linear growth, no
month dummy, no day-of-week, no holiday flag. The only input is today's mean
temperature.

- **Seasonality** is captured, but only *indirectly*, through temperature, which is
  itself seasonal. The model has no notion of "December". It only knows it is cold.
- **Trend or linear growth is not fitted at all.** Nothing in the equation can slope
  upward over years.
- What stands in for a trend is the **refitting**. Because `a`, `b1` and `b2` are
  re-estimated every month under a 6-month half-life, the level drifts as the
  household changes. That is an *adaptive level*, not a fitted trend: it can follow a
  change after it has happened, but it can never project one forward.

So this is not a trend + seasonal + residual decomposition. It is a weather-response
regression, refitted monthly with recency weighting.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(c["month_idx"], c["a"], marker="o", color="#2E86C1")
ax.set_xlabel("fold")
ax.set_ylabel("a, kWh/day at 15 C")
ax.set_title("Household %d - the level is refitted every month, not trended" % H)
plt.show()

---
## 11. What this model is worth

Everything above is how the forecaster was built, not evidence that it works.

The acceptance bar was fixed and dated 2026-08-26, before any weather model was
fitted, and it asks two separate questions. `W2_hl6` answers them very differently,
and that distinction matters more than a single verdict would.

| | measured | daily bar | households meeting it |
|---|---|---|---|
| **accuracy**, median CV(RMSE) | **24.6%** | < 33.1% | **856 of 1,221 (70.1%)** |
| **bias**, median absolute NMBE | **4.646%** | <= 1.7% | **45 of 1,221 (3.7%)** |

**The shortfall is bias, not precision.** The model is reasonably tight around the
right shape and clears the accuracy bar comfortably, on seven households in ten.
What it does not do is sit centred: each household tends to run consistently high or
consistently low, and refitting monthly does not remove that offset. Calling it
"less precise" would point at the wrong half of the table.

Household 756811 above is one of the good ones — a HIGH fit-quality band means
*among the better-fitted*, not *accurate in absolute terms*. Its +0.44%
bias is not what the fleet looks like.

This is the result of record, not a defect to fix in this notebook.